# Offline GPU Embedding — BGE-M3 (Multilingual)

**Prerequisites:**
- Upload `chunk_texts_for_embed.jsonl` as Kaggle Dataset
- Enable GPU accelerator (P100 or T4)

**Model:** `BAAI/bge-m3` — 1024-dim, multilingual, state-of-the-art

**Using `sentence-transformers`** (stable, no FlagEmbedding dependency conflicts)

In [ ]:
!pip install -q -U sentence-transformers
print('Done.')

In [ ]:
import json, os

# Find input file
INPUT = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.jsonl'):
            INPUT = os.path.join(root, f)
            break
if INPUT is None:
    raise FileNotFoundError('No .jsonl found in /kaggle/input')
print(f'Input: {INPUT}')

# Load texts
ids = []
texts = []
with open(INPUT, 'r', encoding='utf-8') as f:
    for line in f:
        rec = json.loads(line)
        ids.append(rec['id'])
        texts.append(rec['text'])
print(f'Loaded {len(texts)} chunks')
print(f'Sample text[:100]: {texts[0][:100]}')

In [ ]:
import time, gc
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'BAAI/bge-m3'
BATCH_SIZE = 64

print(f'Loading {MODEL_NAME} on GPU...')
model = SentenceTransformer(MODEL_NAME)
dim = model.get_sentence_embedding_dimension()
print(f'Model loaded. Embedding dim: {dim}')

# Warmup
_ = model.encode(['warmup'], normalize_embeddings=True)
gc.collect()
print(f'Warmup done. Starting embedding of {len(texts)} chunks...')

t0 = time.time()
all_emb = []

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i+BATCH_SIZE]
    vecs = model.encode(batch, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=False)
    all_emb.append(vecs)
    
    if (i // BATCH_SIZE) % 50 == 0:
        elapsed = time.time() - t0
        done = i + len(batch)
        speed = done / elapsed if elapsed > 0 else 0
        eta = (len(texts) - done) / speed if speed > 0 else 0
        print(f'  {done}/{len(texts)} ({done/len(texts)*100:.0f}%) - {elapsed:.0f}s - {speed:.0f} vec/s - ETA {eta:.0f}s')

elapsed = time.time() - t0
emb_array = np.vstack(all_emb).astype(np.float32)
print(f'\nDone! {emb_array.shape[0]} vectors in {elapsed:.1f}s ({emb_array.shape[0]/elapsed:.0f} vec/s)')
print(f'Shape: {emb_array.shape}, dtype: {emb_array.dtype}')

In [ ]:
# Verify normalization
sample_norms = np.linalg.norm(emb_array[:5], axis=1)
print(f'Sample norms: {sample_norms}')
print(f'All close to 1.0: {np.allclose(sample_norms, 1.0)}')
print(f'Total: {emb_array.shape[0]} vectors x {emb_array.shape[1]} dims')
print(f'Size: {emb_array.nbytes/1024/1024:.0f} MB')

In [ ]:
# Save outputs
np.save('/kaggle/working/embeddings.npy', emb_array)
with open('/kaggle/working/chunk_ids.json', 'w') as f:
    json.dump(ids, f)

print(f'embeddings.npy: {emb_array.shape} ({emb_array.nbytes/1024/1024:.0f} MB)')
print(f'chunk_ids.json: {len(ids)} IDs')
print(f'Model: {MODEL_NAME}')
print(f'\nDownload both files from the Output tab.')